# Unified Model Testing — All Autoencoders

Evaluate **all trained models** (VAE, CNN-AE, LSTM-AE, AAE) against every
CICIDS2017 attack dataset. Produces per-model CSVs and a final ranking.

In [ ]:
import sys, os

# Mount Google Drive and add the project directory to Python path
from google.colab import drive
drive.mount("/content/drive")

NIDS_ROOT = "/content/drive/MyDrive/Colab Notebooks/nids"
sys.path.insert(0, NIDS_ROOT)

import numpy as np
import pandas as pd
import json, glob
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_fscore_support,
)

import joblib
import nids_utils as nu

print(f"TF version: {tf.__version__}")

## 1. Configuration

In [ ]:
# Model families to evaluate and their properties
MODEL_FAMILIES = {
    "vae": {
        "scaler_path": os.path.join(nu.PREPROCESSING_DIR, "scaler_vae.pkl"),
        "input_type": "2d",   # (N, 78)
    },
    "cnnae": {
        "scaler_path": os.path.join(nu.PREPROCESSING_DIR, "scaler_cnnae.pkl"),
        "input_type": "3d",   # (N, 78, 1)
    },
    "lstm_ae": {
        "scaler_path": os.path.join(nu.PREPROCESSING_DIR, "scaler_lstm_ae.pkl"),
        "input_type": "3d",   # (N, 78, 1)
    },
    "aae": {
        "scaler_path": os.path.join(nu.PREPROCESSING_DIR, "scaler_aae.pkl"),
        "input_type": "2d",   # (N, 78)
    },
}

RESULTS_DIR = os.path.join(nu.DRIVE_ROOT, "evaluation_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2. Register Custom Classes (needed for loading .keras files)

In [ ]:
# ---------- CNN-AE classes ----------
@keras.saving.register_keras_serializable()
class CNNEncoder(tf.keras.Model):
    def __init__(self, filters, kernel_size, latent_filters, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.latent_filters = latent_filters
        self.conv_layers = []
        for f in filters:
            self.conv_layers.append(
                layers.Conv1D(f, kernel_size, padding="same", activation="relu")
            )
        self.bottleneck = layers.Conv1D(
            latent_filters, kernel_size, padding="same", activation="relu"
        )
        self.pool = layers.MaxPooling1D(2, padding="same")

    def call(self, x):
        for conv in self.conv_layers:
            x = conv(x)
        x = self.bottleneck(x)
        return self.pool(x)

    def get_config(self):
        return {
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "latent_filters": self.latent_filters,
        }


@keras.saving.register_keras_serializable()
class CNNDecoder(tf.keras.Model):
    def __init__(self, filters, kernel_size, latent_filters, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.latent_filters = latent_filters
        self.deconv_entry = layers.Conv1D(
            latent_filters, kernel_size, padding="same", activation="relu"
        )
        self.upsample = layers.UpSampling1D(2)
        self.conv_layers = []
        for f in reversed(filters):
            self.conv_layers.append(
                layers.Conv1D(f, kernel_size, padding="same", activation="relu")
            )
        self.output_conv = layers.Conv1D(
            1, kernel_size, padding="same", activation="linear"
        )

    def call(self, x):
        x = self.deconv_entry(x)
        x = self.upsample(x)
        for conv in self.conv_layers:
            x = conv(x)
        return self.output_conv(x)

    def get_config(self):
        return {
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "latent_filters": self.latent_filters,
        }


@keras.saving.register_keras_serializable()
class CNNAutoencoder(tf.keras.Model):
    def __init__(self, filters, kernel_size, latent_filters, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.latent_filters = latent_filters
        self.encoder = CNNEncoder(filters, kernel_size, latent_filters)
        self.decoder = CNNDecoder(filters, kernel_size, latent_filters)

    def call(self, x):
        return self.decoder(self.encoder(x))

    def get_config(self):
        return {
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "latent_filters": self.latent_filters,
        }


# ---------- Old CNNAE classes (for loading legacy models) ----------
@keras.saving.register_keras_serializable()
class Encoder(tf.keras.Model):
    def __init__(self, input_shape, latent_dim, **kwargs):
        super().__init__(**kwargs)
        self.input_shape_ = input_shape
        self.latent_dim = latent_dim
        self.conv1 = layers.Conv1D(32, 3, padding="same", activation="relu")
        self.conv2 = layers.Conv1D(latent_dim, 3, padding="same", activation="relu")
        self.maxpool = layers.MaxPooling1D(2, padding="same")

    def call(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return self.maxpool(x)

    def get_config(self):
        return {"input_shape": self.input_shape_, "latent_dim": self.latent_dim}


@keras.saving.register_keras_serializable()
class Decoder(tf.keras.Model):
    def __init__(self, latent_dim, **kwargs):
        super().__init__(**kwargs)
        self.latent_dim = latent_dim
        self.deconv1 = layers.Conv1D(latent_dim, 3, padding="same", activation="relu")
        self.upsample = layers.UpSampling1D(2)
        self.output_layer = layers.Conv1D(1, 3, padding="same", activation="linear")

    def call(self, x):
        x = self.deconv1(x)
        x = self.upsample(x)
        return self.output_layer(x)

    def get_config(self):
        return {"latent_dim": self.latent_dim}


@keras.saving.register_keras_serializable()
class ConvAutoencoder(tf.keras.Model):
    def __init__(self, input_shape, latent_dim, **kwargs):
        super().__init__(**kwargs)
        self.input_shape_ = input_shape
        self.latent_dim = latent_dim
        self.encoder = Encoder(input_shape, latent_dim)
        self.decoder = Decoder(latent_dim)

    def call(self, x):
        return self.decoder(self.encoder(x))

    def get_config(self):
        return {"input_shape": self.input_shape_, "latent_dim": self.latent_dim}


# ---------- VAE Sampling layer ----------
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * eps


print("Custom classes registered.")

## 3. Helper Functions

In [ ]:
FEATURE_COLUMNS = None


def load_and_preprocess(path, scaler, feature_cols):
    """Load a CSV, extract features, scale, and return X + labels."""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    labels = df[nu.LABEL_COLUMN] if nu.LABEL_COLUMN in df.columns else None
    X = df[feature_cols].copy()
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.fillna(X.median(numeric_only=True))
    X_scaled = scaler.transform(X).astype("float32")
    return X_scaled, labels


def reshape_3d(X):
    """Reshape (N, 78) → (N, 78, 1) for CNN/LSTM models."""
    return X.reshape(-1, X.shape[1], 1) if X.ndim == 2 else X


def mse_per_sample(original, reconstructed):
    """Per-sample MSE; works for 2D and 3D arrays."""
    axes = tuple(range(1, original.ndim))
    return np.mean(np.power(original - reconstructed, 2), axis=axes)


def evaluate_model(model, X_benign, attack_sets, threshold, input_type):
    """Run evaluation against all attack sets."""
    reshape_fn = reshape_3d if input_type == "3d" else (lambda x: x)

    X_b = reshape_fn(X_benign)
    recon_benign = model.predict(X_b, verbose=0, batch_size=1024)
    benign_errors = mse_per_sample(X_b, recon_benign)

    rows = []
    for attack_name, X_attack in attack_sets.items():
        X_a = reshape_fn(X_attack)
        recon_atk = model.predict(X_a, verbose=0, batch_size=1024)
        atk_errors = mse_per_sample(X_a, recon_atk)

        y_true = np.concatenate([np.zeros(len(benign_errors)), np.ones(len(atk_errors))])
        y_scores = np.concatenate([benign_errors, atk_errors])
        y_pred = (y_scores > threshold).astype(int)

        prec, rec, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", zero_division=0
        )
        auc_val = roc_auc_score(y_true, y_scores)
        cm = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        acc = (tp + tn) / (tp + tn + fp + fn)

        rows.append({
            "Attack": attack_name,
            "Threshold": threshold,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "AUC": auc_val,
            "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        })

    return pd.DataFrame(rows)

## 4. Discover & Evaluate All Models

In [ ]:
all_summaries = []

for family, family_cfg in MODEL_FAMILIES.items():
    family_dir = os.path.join(nu.MODELS_ROOT, family)
    if not os.path.isdir(family_dir):
        print(f"Skipping {family} — directory not found: {family_dir}")
        continue

    # Load the scaler for this family
    scaler_path = family_cfg["scaler_path"]
    if not os.path.exists(scaler_path):
        print(f"Skipping {family} — scaler not found: {scaler_path}")
        continue

    scaler = joblib.load(scaler_path)

    # Determine feature columns from the scaler
    if hasattr(scaler, 'feature_names_in_'):
        feature_cols = list(scaler.feature_names_in_)
    else:
        # Fallback: load Monday CSV to get columns
        tmp = pd.read_csv(nu.MONDAY_CSV, nrows=1)
        tmp.columns = tmp.columns.str.strip()
        feature_cols = [c for c in tmp.columns if c != nu.LABEL_COLUMN]

    print(f"\n{'#'*60}")
    print(f"# Family: {family}  ({len(feature_cols)} features)")
    print(f"{'#'*60}")

    # Load benign test data
    X_benign, _ = load_and_preprocess(nu.MONDAY_CSV, scaler, feature_cols)

    # Load attack datasets
    attack_sets = {}
    for f in nu.ATTACK_FILES:
        X_atk, _ = load_and_preprocess(f, scaler, feature_cols)
        attack_sets[os.path.basename(f)] = X_atk

    # Iterate over config subdirectories
    for config_dir_name in sorted(os.listdir(family_dir)):
        config_path = os.path.join(family_dir, config_dir_name)
        if not os.path.isdir(config_path):
            continue

        # Find the main .keras model file
        keras_files = [
            f for f in os.listdir(config_path)
            if f.endswith(".keras") and "encoder" not in f
            and "decoder" not in f and "disc" not in f
        ]
        if not keras_files:
            print(f"  No .keras file in {config_path}, skipping.")
            continue

        model_file = keras_files[0]
        model_path = os.path.join(config_path, model_file)

        # Load threshold
        thresh_path = os.path.join(config_path, "threshold.json")
        if os.path.exists(thresh_path):
            with open(thresh_path) as f:
                threshold = json.load(f)["threshold"]
        else:
            threshold = None  # will compute from benign

        print(f"\n  Evaluating: {family}/{config_dir_name}/{model_file}")
        print(f"  Threshold: {threshold}")

        try:
            model = load_model(model_path, compile=False)
        except Exception as e:
            print(f"  LOAD ERROR: {e}")
            continue

        # If no saved threshold, compute from benign data
        if threshold is None:
            reshape_fn = reshape_3d if family_cfg["input_type"] == "3d" else (lambda x: x)
            X_b = reshape_fn(X_benign)
            recon = model.predict(X_b, verbose=0, batch_size=1024)
            errors = mse_per_sample(X_b, recon)
            threshold = float(np.percentile(errors, 97))
            print(f"  Computed threshold (97th pctl): {threshold:.6f}")

        df_results = evaluate_model(
            model, X_benign, attack_sets, threshold, family_cfg["input_type"]
        )

        # Save per-model results
        save_name = f"{family}__{config_dir_name}__eval.csv"
        df_results.to_csv(os.path.join(RESULTS_DIR, save_name), index=False)
        print(df_results[["Attack", "Accuracy", "Recall", "F1", "AUC"]].to_string(index=False))

        # Accumulate summary
        all_summaries.append({
            "Family": family,
            "Config": config_dir_name,
            "Model File": model_file,
            "Mean Accuracy": df_results["Accuracy"].mean(),
            "Mean Recall": df_results["Recall"].mean(),
            "Mean F1": df_results["F1"].mean(),
            "Mean AUC": df_results["AUC"].mean(),
            "Worst Recall": df_results["Recall"].min(),
            "Threshold": threshold,
        })

print(f"\nDone. {len(all_summaries)} model configs evaluated.")

## 5. Final Model Ranking

In [ ]:
if all_summaries:
    summary = pd.DataFrame(all_summaries)
    summary = summary.sort_values(
        by=["Mean Recall", "Mean F1", "Mean AUC"], ascending=False
    )

    summary_path = os.path.join(RESULTS_DIR, "MODEL_COMPARISON_SUMMARY.csv")
    summary.to_csv(summary_path, index=False)

    print("\n" + "=" * 70)
    print("  FINAL MODEL RANKING")
    print("=" * 70)
    print(summary.to_string(index=False))
else:
    print("No models were evaluated.")

## 6. Visualization — Comparison Heatmap

In [ ]:
if all_summaries:
    metrics = ["Mean Accuracy", "Mean Recall", "Mean F1", "Mean AUC"]
    plot_df = summary.set_index("Config")[metrics]

    fig, ax = plt.subplots(figsize=(10, max(4, len(plot_df) * 0.6)))
    sns.heatmap(plot_df, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax)
    ax.set_title("Model Comparison Heatmap")
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, "comparison_heatmap.png"), dpi=150)
    plt.show()

## 7. Visualization — Per-attack Recall Comparison

In [ ]:
if all_summaries:
    # Collect all per-model eval CSVs for a grouped bar chart
    all_evals = []
    for f in sorted(os.listdir(RESULTS_DIR)):
        if f.endswith("__eval.csv"):
            df_e = pd.read_csv(os.path.join(RESULTS_DIR, f))
            model_label = f.replace("__eval.csv", "")
            df_e["Model"] = model_label
            all_evals.append(df_e)

    if all_evals:
        combined = pd.concat(all_evals, ignore_index=True)
        pivot = combined.pivot(index="Attack", columns="Model", values="Recall")

        fig, ax = plt.subplots(figsize=(14, 6))
        pivot.plot(kind="bar", ax=ax, width=0.85)
        ax.set_ylabel("Recall")
        ax.set_title("Per-Attack Recall — All Models")
        ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=7)
        plt.xticks(rotation=30, ha="right", fontsize=8)
        fig.tight_layout()
        fig.savefig(os.path.join(RESULTS_DIR, "per_attack_recall.png"), dpi=150)
        plt.show()